<span style="color:#246485;font-weight:700;font-size:32px"> 
Option Analysis & Scoring Pipeline
</span> <br>

#### **architected by --> Alf Haugen**
-----

#### **Overview:** 
This notebook runs the end-to-end options analysis and scoring pipeline; 
from raw API data capture, premium and risk data build and scoring to a final 
scored universe structured for dashboard visualzation. 

The pipeline scans a universe of ~600 NYSE and Nasdaq symbols on a user-defined date, 
building a structured dataset of options premiums, implied volatility, historical volatility, 
price spike activity, and metrics across the forward term structure. 
All metrics are then scored and ranked into a two-dimensional premium/risk framework for put-selling alpha opportunities. 

#### **Objectives:** 
- Pull historical options chain and daily time-series price data from Alpha Vantage by symbol
- Build normalized put premium metrics across 5 DTE windows (Days to Expiration) 2-week, 1-month, 45-day, then the next two future
  LEAPS (14, 30, 45, over60_1, over60_2) and 4 delta-based strike buckets (ATM, Slight, Moderate, Far)
- Compute ATM straddle premium and three premium efficiency metrics
  (vs theoretical move, vs IV, vs realized vol)
- Compute Historical Volatility (HV_20, HV_30, HV_60) and IV/HV ratios per DTE window
- Compute spike analysis across 30 and 60-day windows with inward symbol and
  universal relative signals, producing a blended spike score based on frequency and
  magnitude
- Compute relative volatility vs SPY and QQQ benchmarks
- Score the universe on two axes — Premium Score and Risk Score,
  using weighted composites of the above metrics
- Assign each symbol to one of four quadrants (Q1 High Premium/Low Risk → Q4) for visualization
  and analysis
- Compute term structure slopes for premium and IV across DTE windows
- Output a single flat CSV per run for dashboard consumption

#### **Dataset Configurations:** 
- **Data source:**       Alpha Vantage Premium API (HISTORICAL_OPTIONS + TIME_SERIES_DAILY)
- **Option date:**       option_date  — the historical chain snapshot date (e.g. '2026-02-27')
- **HV as-of date:**     as_of_date   — price history for look-back analysis, truncated to prevent lookahead
- **Universe:**          ~600 symbols across NYSE and Nasdaq (ticker_list)
- **DTE windows:**       14-day (Friday-snapped), 30-day, over60_1, over60_2
- **Strike buckets:**    Delta based at: ATM (delta 0.40-0.60), Slight (0.25-0.40),
                         Moderate (0.15-0.25), Far (0.05-0.15)
- **Rate limit:**        75 calls/min by Alpha Vantage → batches of 37 symbols with 61s pause between batches
- **Benchmarks:**        SPY and QQQ HV_30 fetched once before the loop for relative vol

#### **Model Architecture Overview** 
The pipeline runs in four layers:

  1. API Layer (av_api_calls.py)
     Fetches options chain and daily prices per symbol. Manages rate limiting,
     error handling, and benchmark HV computation for SPY/QQQ.

  2. Premium Layer (option_prem_iv_builder.py)
     Parses and filters contracts by delta bucket and liquidity.
     Computes normalized premium (extrinsic/spot), aggregates by DTE/bucket,
     builds ATM straddle, and computes three premium efficiency metrics.

  3. Risk Layer (hist_vol_iv_risk_builder.py)
     Computes HV across three windows, extracts ATM IV per DTE window,
     builds IV/HV ratios with categorical signals, and runs spike analysis
     across 30 and 60-day windows with self-relative and universe signals.

  4. Scoring Layer (score_universe.py)
     Standardizes and weights premium and risk components into composite scores.
     Computes term structure regression slopes, premium efficiency signals,
     universe-relative spike and HV percentile ranks, and assigns quadrant labels.
     Outputs the final scored master DataFrame.

See ReadMe for further details on overall scoring model design. 

#### **Expected Results** 
- option_analysis_master  : flat DataFrame, one row per symbol, ~100+ columns
                            covering all premium, IV, HV, spike, and efficiency metrics.
                            ---> Pushes into Scoring Layer 
- scored_master           : option_analysis_master enriched with premium_score,
                            risk_score, quadrant, term structure slopes,
                            prem_efficiency_signal per DTE, blended spike_signal_universe,
                            HV_30_pct, relative_vol_spy_pct, relative_vol_qqq_pct
- error_log_df            : DataFrame of failed symbols with error message and
                            AV response detail for diagnosis
- CSV output              : scored_master saved to disk for dashboard consumption,
                            filename includes the option_date for run tracking

---

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from datetime import datetime, timedelta
import time
from IPython.display import clear_output
from typing import Optional
%config InlineBackend.figure_format = 'retina'
from scipy import stats

import requests
import json
import os
import dotenv
dotenv.load_dotenv()
import sys
import traceback
sys.tracebacklimit = 0 # turn off the error tracebacks

### reload amended source code / .py
%load_ext autoreload
%autoreload 2

In [2]:
os.getcwd()
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(project_root)

src_path = os.path.join(project_root, 'src')
sys.path.insert(0, src_path)

print(src_path)  # verify path looks right

/Users/AlfHaugen/Python/Code/options-alpha-scanner
/Users/AlfHaugen/Python/Code/options-alpha-scanner/src


In [3]:
### Imports
import av_api_calls
import score_universe

In [5]:
### symbols
path_one = src_path + '/tickers.txt'
# path_one = src_path + '/tickers_one_hun.txt'
# path_one = src_path + '/tickers_two_hun.txt'
# path_one = src_path + '/tickers_six_hun.txt'

with open(path_one, 'r') as f:
    ticker_list = [item.strip() for item in f.read().split(',')]

print(f' The number of symbols is {len(ticker_list)}')
ticker_list[-5:]

 The number of symbols is 10


['TQQQ', 'META', 'TSLA', 'CB', 'RJF']

In [6]:
# Setting up request and params. 
API_KEY = os.getenv('MktPremiumKey')

In [7]:
#### Data Params --->
option_date = '2026-03-27'   # date for historical options chain
as_of_date  = '2026-03-27'   # truncate HV history at this date

#### **Naming Convention**

In [8]:
### Naming --> 
file_size = '10'
data_as_of = as_of_date
opt_file_name = 'opt_analysis_master_' + file_size + '_' + data_as_of + '.csv'
error_log_file_name = 'error_log_' + file_size + '_' + data_as_of + '.csv'
stats_log_file_name = 'stats_desc_' + file_size + '_' + data_as_of + '.csv'
field_error_file_name = 'field_errors_' + file_size + '_' + data_as_of + '.csv'
opt_file_name

'opt_analysis_master_10_2026-03-27.csv'

--------
### **Run the API / Option Scanner Data Build**

----

In [9]:
### Call Option Scanner -->
option_analysis_master, error_log_df = av_api_calls.option_analysis_scan(ticker_list, API_KEY, option_date, as_of_date)

Scanning 10 symbols
[10/10] Current: RJF
Succeeded : 9
Failed    : 0
API calls : 18
Elapsed   : 00:22
 Pushing in the spot value 141.32 
Building Option Premium Data
  ⚠️ No expiration found near over60_1 DTE for RJF
  ⚠️ No expiration found near over60_1 DTE for RJF
Building Historical Vol and Spike Data
Building Straddle Data
OK
------------------------------------------

Done: 10 succeeded, 0 failed
Total time: 00:23
Master shape: (10, 113)


In [10]:
print(option_analysis_master.shape)
option_analysis_master.head(5)

(10, 113)


,symbol,date,spot,expiration_14,premium_atm_14,iv_atm_14,premium_slight_14,iv_slight_14,premium_moderate_14,iv_moderate_14,premium_far_14,iv_far_14,expiration_30,premium_atm_30,iv_atm_30,premium_slight_30,iv_slight_30,premium_moderate_30,iv_moderate_30,premium_far_30,iv_far_30,expiration_45,premium_atm_45,iv_atm_45,premium_slight_45,iv_slight_45,premium_moderate_45,iv_moderate_45,premium_far_45,iv_far_45,expiration_over60_1,premium_atm_over60_1,iv_atm_over60_1,premium_slight_over60_1,iv_slight_over60_1,premium_moderate_over60_1,iv_moderate_over60_1,premium_far_over60_1,iv_far_over60_1,expiration_over60_2,premium_atm_over60_2,iv_atm_over60_2,premium_slight_over60_2,iv_slight_over60_2,premium_moderate_over60_2,iv_moderate_over60_2,premium_far_over60_2,iv_far_over60_2,straddle_14,put_atm_14,call_atm_14,prem_per_iv_primary_14,prem_per_iv_sec_14,prem_per_hv30_14,straddle_30,put_atm_30,call_atm_30,prem_per_iv_primary_30,prem_per_iv_sec_30,prem_per_hv30_30,straddle_45,put_atm_45,call_atm_45,prem_per_iv_primary_45,prem_per_iv_sec_45,prem_per_hv30_45,straddle_over60_1,put_atm_over60_1,call_atm_over60_1,prem_per_iv_primary_over60_1,prem_per_iv_sec_over60_1,prem_per_hv30_over60_1,straddle_over60_2,put_atm_over60_2,call_atm_over60_2,prem_per_iv_primary_over60_2,prem_per_iv_sec_over60_2,prem_per_hv30_over60_2,HV_20,HV_30,HV_60,atm_iv_14,ratio_14,spread_14,signal_14,atm_iv_30,ratio_30,spread_30,signal_30,atm_iv_45,ratio_45,spread_45,signal_45,atm_iv_over60_1,ratio_over60_1,spread_over60_1,signal_over60_1,atm_iv_over60_2,ratio_over60_2,spread_over60_2,signal_over60_2,spike_count_30,spike_ratio_30,avg_spike_pct_30,max_spike_pct_30,spike_signal_30,spike_count_60,spike_ratio_60,avg_spike_pct_60,max_spike_pct_60,spike_signal_60,relative_vol_spy,relative_vol_qqq
0,GME,2026-03-27,22.10,2026-04-10,2.4661,39.0480,1.6063,36.6090,0.9729,41.4865,0.4186,45.3895,2026-04-17,3.0430,38.0720,1.9570,38.5600,1.1765,41.4870,0.5882,43.9255,2026-05-15,4.4344,39.0480,3.1448,38.5600,1.8326,39.5360,0.8710,42.9505,2026-06-18,5.8899,43.1130,4.5701,43.4380,2.6471,45.3890,1.3725,49.9420,2026-07-17,6.7949,41.8117,5.4072,41.9745,2.6244,42.9505,1.0784,47.0153,4.48,2.47,2.01,0.4867,0.0632,0.0986,5.85,3.04,2.81,0.5321,0.0799,0.1217,7.64,4.43,3.20,0.4435,0.1136,0.1774,9.86,5.89,3.97,0.3987,0.1366,0.2356,11.86,6.79,5.06,0.4253,0.1625,0.2718,0.2634,0.2500,0.3711,0.3905,1.562,0.1405,Very Rich Vol,0.3807,1.523,0.1307,Very Rich Vol,0.3905,1.562,0.1405,Very Rich Vol,0.4311,1.724,0.1811,Very Rich Vol,0.4181,1.672,0.1681,Very Rich Vol,1,0.73,3.75,3.75,Normal,3,1.10,6.56,7.93,Normal,1.78,1.41
1,AGQ,2026-03-27,103.43,2026-04-10,9.3082,148.1914,7.9095,160.3581,4.3991,164.6545,1.7484,172.8657,2026-04-17,10.7082,142.8016,9.2365,151.5980,5.2270,166.6060,2.2237,178.8005,2026-05-15,14.6553,143.3782,15.5145,144.1090,8.0610,149.2890,2.1512,145.8750,2026-06-18,17.2713,142.0916,20.4516,136.3438,9.6482,136.2811,4.4039,140.4114,2026-09-18,21.4295,137.9102,28.8669,128.5995,16.5893,122.9482,8.4598,122.9485,14.63,9.31,5.32,0.4187,0.0628,0.0701,16.66,10.71,5.95,0.4041,0.0750,0.0806,19.14,14.66,4.48,0.3027,0.1022,0.1103,19.70,17.27,2.43,0.2416,0.1216,0.1300,21.75,21.43,0.32,0.1892,0.1554,0.1613,1.2394,1.3284,2.5001,1.4819,1.116,0.1535,Equiv. Vol,1.4280,1.075,0.0996,Equiv. Vol,1.4338,1.079,0.1054,Equiv. Vol,1.4209,1.070,0.0925,Equiv. Vol,1.3791,1.038,0.0507,Equiv. Vol,1,0.73,17.98,17.98,Normal,2,0.73,63.86,91.40,Normal,9.48,7.51
2,QQQ,2026-03-27,562.58,2026-04-10,1.9816,30.8127,1.5464,33.0728,0.7406,35.9583,0.3426,40.3485,2026-04-17,2.3358,30.0236,1.7470,33.4873,1.0517,36.0885,0.4633,42.0288,2026-05-15,3.2400,28.5441,2.6242,32.4038,1.6510,35.7417,0.6813,43.2879,2026-06-18,3.9694,27.2432,3.3462,31.1102,1.9350,35.2268,0.8987,42.1520,2026-08-21,4.6114,26.7554,4.3649,30.2133,2.6017,33.8990,1.2283,40.6196,3.76,1.98,1.78,0.5183,0.0643,0.1120,4.49,2.34,2.16,0.5185,0.0778,0.1320,6.18,3.24,2.94,0.4908,0.1135,0.1831,7.42,3.97,3.45,0.4746,0.1457,0.2243,8.23,4.61,3.62,0.4028,0.1724,0.2606,0.1883,0.1770,0.1707,0.308

-----
### **Error Log and DF Stats**
- Review Results logged during the API Loop Run

In [ ]:
### Reviewing the Errors logged
print(error_log_df.shape)
error_log_df.head(10)

In [ ]:
### Save to CSV -->
error_log_df.to_csv(error_log_file_name, index=True)

#### **Statistics Review**

In [ ]:
### Run Description Stats
data_describe = option_analysis_master.describe().T
# data_describe.head()

In [ ]:
### Save to CSV -->
data_describe.to_csv(stats_log_file_name, index=True)

#### **Field Specific Review**

In [ ]:
field_nan_report = av_api_calls.audit_non_numeric(option_analysis_master)
print(field_nan_report['symbol'].unique())
field_nan_report.head(5)

In [ ]:
### Save to CSV --> 
field_nan_report.to_csv(field_error_file_name, index=True)

----
### **Run block to Save Premium / Risk Data to CSV**

In [ ]:
### Write to CSV -->
option_analysis_master.to_csv(opt_file_name, index=False)

<hr style="border:7px solid #246485">

## **Scoring Module**
- Push the Option Premium and Historical Vol. Builder dataframe into the Scoring module. 

In [ ]:
### If loading past data from saved CSV.....
# option_analysis_master = pd.read_csv('opt_analysis_master_200_sym_2026_02_27.csv')
# print(option_analysis_master.shape)
# option_analysis_master.head()

In [ ]:
### Run data through Scoring Module: ----------->
scoring_data = score_universe.score_universe(option_analysis_master)

In [ ]:
print(scoring_data.shape)
scoring_data.sort_values('risk_score',ascending=False).head(4)

#### **Save to CSV**

In [ ]:
### Write to CSV -->
opt_file_name = 'option_scores_master' + '_' + file_size + '_' + data_as_of + '.csv'
scoring_data.to_csv(opt_file_name, index=False)

### **End**
----